In [ ]:
%load_ext autoreload
%autoreload 2

# Custom source

Subclass `BaseSource`, implement `get_allocations`, and the registry picks
the workload up automatically — the inherited `get_pool`/`get_memory`/
`get_system` build the entity hierarchy, and the benchmark harness accepts
it by name.

In [ ]:
import random

import omnimalloc as om
from omnimalloc.benchmark.sources import BaseSource, available_sources

`get_allocations(num_allocations, skip)` returns a deterministic tuple;
`skip` discards that many leading allocations so the base class can carve
consecutive, non-repeating batches for multi-pool workloads.

This source models the activations of a sequential layer chain: layer `i`
writes its output at time `i` and the consumer at layer `c` reads it while
running, so the tensor lives `[i, c + 1)`. Most outputs feed the next layer;
occasional skip connections reach `skip_distance` layers ahead and stay live
in between — the long thin rectangles a good packer must thread around.

In [ ]:
class LayerChainSource(BaseSource):
    """Activations of a layer chain with occasional skip connections."""

    def __init__(
        self,
        num_allocations: int = 100,
        size_min: int = 1 << 10,
        size_max: int = 1 << 20,
        skip_distance: int = 8,
        skip_probability: float = 0.2,
        seed: int | None = 42,
    ) -> None:
        super().__init__(num_allocations=num_allocations)
        self.size_min = size_min
        self.size_max = size_max
        self.skip_distance = skip_distance
        self.skip_probability = skip_probability
        self.seed = seed

    def get_allocations(
        self, num_allocations: int | None = None, skip: int = 0
    ) -> tuple[om.Allocation, ...]:
        total = num_allocations if num_allocations is not None else self.num_allocations
        rng = random.Random(self.seed)
        for _ in range(skip):
            self._generate_one(rng, 0)
        return tuple(self._generate_one(rng, skip + i) for i in range(total))

    def _generate_one(self, rng: random.Random, layer: int) -> om.Allocation:
        is_skip = rng.random() < self.skip_probability
        consumer = layer + (self.skip_distance if is_skip else 1)
        return om.Allocation(
            id=layer,
            size=rng.randint(self.size_min, self.size_max),
            start=layer,
            end=consumer + 1,
        )

Defining the class registered it as `"layer_chain"` (the `Source` role token
is stripped). Draw a pool, allocate it, and plot the placement.

In [ ]:
assert "layer_chain" in available_sources()

pool = LayerChainSource(num_allocations=200).get_pool()
pool = om.allocate(pool, validate=True)
print(
    f"peak memory {pool.size}, lower bound {pool.pressure}, "
    f"efficiency {pool.efficiency:.3f}"
)
om.plot_allocation(pool)

Registered sources plug straight into the benchmark harness by name.

In [ ]:
from omnimalloc.benchmark import plot_benchmark, run_benchmark

campaign = run_benchmark(
    allocators=("naive", "greedy_by_size", "omni"),
    sources=("layer_chain",),
    variants=(50, 100, 200, 500),
    validate=True,
)
plot_benchmark(campaign)